# Pipeline v2 Apps Workbench

Notebook orchestrator for the current `src.pipeline_v2` app-style modules.

Each pure app stays JSON/dict-in and JSON/dict-out. This notebook builds app inputs, calls app entrypoints, and writes notebook-owned outputs under `output_v2/`. Voiceover is app-like but writes WAV files under its caller-provided output directory because audio is its core product.


In [ ]:
from __future__ import annotations

from copy import deepcopy
from datetime import datetime, timedelta, timezone
import json
from pathlib import Path
import sys

import yaml

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "pipeline_v2" / "apps").exists():
    raise RuntimeError("Run this notebook from the repository root or output_v2 directory.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

APPS_DIR = PROJECT_ROOT / "src" / "pipeline_v2" / "apps"
NEWS_COLLECTION_DIR = APPS_DIR / "news_collection"
OUTPUT_V2 = PROJECT_ROOT / "output_v2"
IST = timezone(timedelta(hours=5, minutes=30))

print(f"Project root: {PROJECT_ROOT}")
print(f"Apps dir:     {APPS_DIR}")
print(f"Output root:  {OUTPUT_V2}")

In [ ]:
# Change these values for a different v2 app run.
RUN_PROFILE = "daily_48h_all_sources"
RUN_ID = f"v2_apps_{datetime.now(IST).strftime('%Y_%m_%d_%H%M%S')}_{RUN_PROFILE}"

NEWS_OUTPUT_DIR = OUTPUT_V2 / "news_collection" / RUN_ID
EVIDENCE_OUTPUT_DIR = OUTPUT_V2 / "evidence_prep" / RUN_ID
STORY_OUTPUT_DIR = OUTPUT_V2 / "story_mapping" / RUN_ID
VIDEO_OUTPUT_DIR = OUTPUT_V2 / "video_packaging" / RUN_ID
SCRIPT_OUTPUT_DIR = OUTPUT_V2 / "script_generation" / RUN_ID
METADATA_OUTPUT_DIR = OUTPUT_V2 / "metadata_creator" / RUN_ID
HANDOFF_OUTPUT_DIR = OUTPUT_V2 / "handoffs" / RUN_ID

VOICEOVER_MODEL_CACHE_ROOT = OUTPUT_V2 / "model_cache"
VOICEOVER_F5_REF_AUDIO_PATH = OUTPUT_V2 / "voiceover" / "_smoke_chatterbox_no_xet" / "audio" / "full_voiceover.wav"
VOICEOVER_F5_REF_TEXT = "Today, we are testing a more expressive local voiceover model for the new pipeline."
VOICEOVER_INPUT_DIR = OUTPUT_V2 / "voiceover_inputs"

# Voiceover is fully driven by this config envelope. Change provider, style,
# speed, chunking, reference audio, or model settings here before running App 7.
def voiceover_variant(config: dict) -> str:
    def slug(value: object) -> str:
        text = str(value).strip().lower().replace(".", "p")
        text = "".join(ch if ch.isalnum() else "_" for ch in text)
        return "_".join(part for part in text.split("_") if part)

    speed = config.get("speed")
    speed_label = f"speed_{slug(speed)}" if speed is not None else "style_speed"
    voice_label = slug(config.get("voice") or "default_voice")
    return "_".join(
        part for part in [slug(config.get("provider", "provider")), slug(config.get("style", "style")), voice_label, speed_label] if part
    )

VOICEOVER_CONFIG = {
    "provider": "kokoro_local",  # kokoro_local | chatterbox_local | f5_tts_local | dia_transformers
    "style": "general_youtube",
    "voice": None,
    "speed": 0.98,
    "language_code": "a",
    "sample_rate_hz": 24000,
    "keep_chunks": True,
    "force": True,
    "output_dir": "",  # filled from RUN_ID below unless overridden
    "chunking": {
        "chunk_max_chars": None,
        "max_sentences_per_chunk": None,
        "chunk_pause_seconds": None,
        "paragraph_pause_seconds": None,
    },
    "chatterbox": {
        "device": "auto",
        "exaggeration": 0.65,
        "cfg_weight": 0.5,
        "audio_prompt_path": None,
        "cache_dir": str(VOICEOVER_MODEL_CACHE_ROOT / "huggingface_local"),
    },
    "f5_tts": {
        "model": "F5TTS_v1_Base",
        "device": "cpu",
        "ref_audio_path": str(VOICEOVER_F5_REF_AUDIO_PATH),
        "ref_text": VOICEOVER_F5_REF_TEXT,
        "cache_dir": str(VOICEOVER_MODEL_CACHE_ROOT / "f5_tts"),
        "nfe_step": 16,
        "cfg_strength": 2.0,
        "sway_sampling_coef": -1.0,
        "target_rms": 0.1,
        "cross_fade_duration": 0.15,
        "remove_silence": False,
        "seed": 1234,
    },
    "dia": {
        "model_checkpoint": "nari-labs/Dia-1.6B-0626",
        "device": "cpu",
        "cache_dir": str(VOICEOVER_MODEL_CACHE_ROOT / "dia"),
        "max_new_tokens": 256,
        "guidance_scale": 3.0,
        "temperature": 1.8,
        "top_p": 0.9,
        "top_k": 45,
        "speaker_tag": "S1",
    },
}
VOICEOVER_OUTPUT_DIR = OUTPUT_V2 / "voiceover" / f"{RUN_ID}_{voiceover_variant(VOICEOVER_CONFIG)}"
VOICEOVER_CONFIG["output_dir"] = str(VOICEOVER_OUTPUT_DIR)

print(f"Run profile: {RUN_PROFILE}")
print(f"Run ID:      {RUN_ID}")
print(f"Voiceover:   {VOICEOVER_CONFIG['provider']} -> {VOICEOVER_CONFIG['output_dir']}")


In [ ]:
def load_yaml(path: Path) -> dict:
    data = yaml.safe_load(path.read_text(encoding="utf-8"))
    if not isinstance(data, dict):
        raise ValueError(f"Expected YAML object at {path}")
    return data


def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")


def write_jsonl(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    text = "\n".join(json.dumps(row, ensure_ascii=False) for row in rows)
    if text:
        text += "\n"
    path.write_text(text, encoding="utf-8")


def write_handoff_files(files: list[dict], output_dir: Path) -> list[str]:
    output_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    for file_info in files:
        filename = file_info["filename"]
        if "/" in filename or "\\" in filename:
            raise ValueError(f"Unsafe handoff filename: {filename}")
        path = output_dir / filename
        path.write_text(file_info["content"], encoding="utf-8")
        paths.append(str(path))
    return paths


def require_app_ready(result: dict, app_name: str, *, allowed_statuses=("ok",), handoff_key=None) -> None:
    status = result.get("status")
    if status not in allowed_statuses:
        summary = result.get("summary", {})
        failure_reason = summary.get("failure_reason") or summary.get("error") or result.get("reason") or "No failure reason provided."
        raise RuntimeError(f"{app_name} is not ready. status={status!r}. reason={failure_reason}")
    if handoff_key and not bool(result.get("handoff", {}).get(handoff_key, False)):
        summary = result.get("summary", {})
        failure_reason = summary.get("failure_reason") or summary.get("error") or f"handoff.{handoff_key} is false"
        raise RuntimeError(f"{app_name} is not ready. status={status!r}. reason={failure_reason}")


def require_non_empty_script(script_generation_result: dict) -> str:
    require_app_ready(script_generation_result, "script_generation", handoff_key="ready_for_verification")
    script_package = script_generation_result.get("data", {}).get("script_package", {})
    full_script = str(script_package.get("full_script", "")).strip()
    if not full_script:
        checks = script_package.get("checks", {})
        errors = checks.get("errors") or []
        reason = "; ".join(str(error) for error in errors) or "script_package.full_script is empty"
        raise RuntimeError(f"script_generation did not produce a non-empty full_script. reason={reason}")
    return full_script


def collection_window_for_profile(profile: dict) -> dict:
    end = datetime.now(IST)
    lookback_hours = int(profile["lookback_hours"])
    start = end - timedelta(hours=lookback_hours)
    return {
        "start": start.isoformat(),
        "end": end.isoformat(),
        "lookback_hours": lookback_hours,
    }


def build_news_collection_input(run_id: str, run_profile: str) -> dict:
    profile_config = load_yaml(NEWS_COLLECTION_DIR / "profiles.yaml")
    source_config = load_yaml(NEWS_COLLECTION_DIR / "source_config.yaml")
    profiles = profile_config["profiles"]
    if run_profile not in profiles:
        raise KeyError(f"Unknown run profile: {run_profile}")

    profile = profiles[run_profile]
    source_groups = source_config["layer_1"]["source_groups"]
    window = collection_window_for_profile(profile)
    collectors = {}
    for group_name in profile["enabled_source_groups"]:
        collector_config = deepcopy(source_groups[group_name])
        collector_config["source_group"] = group_name
        collector_config["collection_window"] = window
        collector_config["query_themes"] = list(profile["query_themes"])
        collector_config["runtime"] = deepcopy(profile.get("collector_caps", {}).get(group_name, {}))
        collectors[group_name] = collector_config

    return {
        "run": {
            "run_id": run_id,
            "generated_at": datetime.now(IST).isoformat(),
        },
        "config": {
            "enabled_source_groups": list(profile["enabled_source_groups"]),
            "query_themes": list(profile["query_themes"]),
            "collectors": collectors,
        },
        "input": {
            "run_profile": run_profile,
            "channel": deepcopy(profile_config["channel"]),
        },
    }


def build_evidence_prep_input(run_id: str, news_input: dict, news_result: dict) -> dict:
    first_collector = next(iter(news_input["config"]["collectors"].values()))
    return {
        "run": {
            "run_id": run_id,
            "generated_at": datetime.now(IST).isoformat(),
        },
        "config": {
            "collection_window": deepcopy(first_collector["collection_window"]),
            "query_themes": list(news_input["config"]["query_themes"]),
            "max_research_papers": 12,
            "filter_policy": {},
        },
        "input": {
            "news_collection": news_result,
        },
    }


def build_story_mapping_input(run_id: str, evidence_input: dict, evidence_result: dict) -> dict:
    return {
        "run": {
            "run_id": run_id,
            "generated_at": datetime.now(IST).isoformat(),
        },
        "config": {
            "embedding_model": "all-MiniLM-L6-v2",
            "collection_window": deepcopy(evidence_input["config"]["collection_window"]),
            "use_llm_story_labels": True,
            "clustering": {
                "max_pair_hours": None,
                "semantic_primary_threshold": 0.80,
                "semantic_supported_threshold": 0.78,
                "semantic_title_anchor_threshold": 0.74,
                "semantic_merge_groups": [
                    "ai_company_blogs",
                    "tech_news_apis",
                    "youtube_podcast_transcripts",
                ],
            },
            "story_label": {
                "provider": "openai",
                "model": "gpt-5-nano",
                "prompt_version": "v1",
                "worker_count": 10,
                "temperature": 0.15,
                "max_records_per_story": 6,
            },
        },
        "input": {
            "evidence_prep": evidence_result,
        },
    }


def build_video_packaging_input(run_id: str, story_mapping_result: dict) -> dict:
    return {
        "run": {
            "run_id": run_id,
            "generated_at": datetime.now(IST).isoformat(),
        },
        "config": {
            "target_format": "roundup",
            "roundup": {
                "candidate_limit": 20,
                "min_stories": 3,
                "max_stories": 7,
                "target_duration_seconds": 480,
                "intro_seconds": 35,
                "closing_seconds": 20,
                "role_weights": {
                    "lead": 1.35,
                    "major": 1.05,
                    "brief": 0.75,
                },
                "max_per_entity": 2,
                "max_per_product": 1,
                "max_per_event_family": 2,
                "allowed_priority_buckets": ["high", "medium"],
                "exclude_noise": True,
                "exclude_tutorials": True,
            },
        },
        "input": {
            "story_mapping": story_mapping_result,
        },
    }


def build_script_generation_input(run_id: str, video_packaging_result: dict) -> dict:
    return {
        "run": {
            "run_id": run_id,
            "generated_at": datetime.now(IST).isoformat(),
        },
        "config": {
            "provider": "openai",
            "model": "gpt-5-nano",
            "prompt_version": "v1",
            "temperature": 0.4,
            "words_per_minute": 150,
            "duration_tolerance_seconds": 45,
            "worker_count": 1,
            "script_style": {
                "audience": "AI-curious professionals",
                "tone": "clear, energetic, grounded",
                "avoid": [
                    "dry technical reporting",
                    "overhype",
                    "fake certainty",
                    "source-by-source summaries",
                ],
            },
            "source_policy": {
                "use_selected_stories_only": True,
                "include_full_records": True,
            },
            "reflection": {
                "enabled": True,
                "max_revision_passes": 1,
                "target_score": 8,
            },
        },
        "input": {
            "video_packaging": video_packaging_result,
        },
    }


def build_metadata_creator_input(run_id: str, script_generation_result: dict, video_packaging_result: dict) -> dict:
    profile_config = load_yaml(NEWS_COLLECTION_DIR / "profiles.yaml")
    channel = deepcopy(profile_config.get("channel", {}))
    return {
        "run": {
            "run_id": run_id,
            "generated_at": datetime.now(IST).isoformat(),
        },
        "config": {
            "provider": "openai",
            "model": "gpt-5-nano",
            "prompt_version": "v1",
            "temperature": 0.4,
            "channel": {
                "channel_name": channel.get("channel_name", "ManiVerse - AI"),
                "content_promise": channel.get("content_promise", ""),
            },
            "metadata_policy": {
                "title_option_count": 8,
                "thumbnail_text_option_count": 5,
                "max_tags": 20,
                "include_source_urls": True,
            },
            "memory_context": {
                "recent_title_angles": [],
            },
        },
        "input": {
            "script_generation": script_generation_result,
            "video_packaging": video_packaging_result,
        },
    }


def build_voiceover_input(run_id: str, script_generation_result: dict, config_override=None) -> dict:
    config = deepcopy(config_override or VOICEOVER_CONFIG)
    if not str(config.get("output_dir", "")).strip():
        config["output_dir"] = str(OUTPUT_V2 / "voiceover" / f"{run_id}_{voiceover_variant(config)}")
    return {
        "run": {
            "run_id": run_id,
            "generated_at": datetime.now(IST).isoformat(),
        },
        "config": config,
        "input": {
            "script_generation": script_generation_result,
        },
    }


def resolved_voiceover_chunking(config: dict) -> dict:
    from src.pipeline_v2.voiceover.config import resolve_style

    style_profile = resolve_style(config["style"])
    chunking = config.get("chunking") or {}
    return {
        "max_chars": chunking.get("chunk_max_chars") or style_profile.chunk_max_chars,
        "max_sentences_per_chunk": chunking.get("max_sentences_per_chunk") or style_profile.max_sentences_per_chunk,
        "chunk_pause_seconds": (
            chunking.get("chunk_pause_seconds")
            if chunking.get("chunk_pause_seconds") is not None
            else style_profile.chunk_pause_seconds
        ),
        "paragraph_pause_seconds": (
            chunking.get("paragraph_pause_seconds")
            if chunking.get("paragraph_pause_seconds") is not None
            else style_profile.paragraph_pause_seconds
        ),
    }


def build_v2_manifest_context(
    run_id: str,
    run_profile: str,
    news_input: dict,
    evidence_input: dict,
    story_mapping_input: dict,
    video_packaging_input: dict,
    script_generation_input: dict,
    metadata_creator_input: dict,
) -> dict:
    channel = deepcopy(news_input.get("input", {}).get("channel", {}))
    return {
        "run_id": run_id,
        "run_date": datetime.now(IST).date().isoformat(),
        "generated_at": datetime.now(IST).isoformat(),
        "pipeline_track": "pipeline_v2",
        "orchestrator": "notebooks/v2_apps_workbench.ipynb",
        "layer_0": {
            "run_profile": run_profile,
            "collection_window": deepcopy(evidence_input["config"]["collection_window"]),
            "channel": channel,
        },
        "layer_1": {
            "enabled_source_groups": list(news_input["config"]["enabled_source_groups"]),
            "query_themes": list(news_input["config"]["query_themes"]),
        },
        "layer_5": {
            "target_format": video_packaging_input["config"]["target_format"],
        },
        "apps": {
            "news_collection": {"config": deepcopy(news_input["config"])},
            "evidence_prep": {"config": deepcopy(evidence_input["config"])},
            "story_mapping": {"config": deepcopy(story_mapping_input["config"])},
            "video_packaging": {"config": deepcopy(video_packaging_input["config"])},
            "script_generation": {"config": deepcopy(script_generation_input["config"])},
            "metadata_creator": {"config": deepcopy(metadata_creator_input["config"])},
        },
    }


def build_summary_bundle(
    news_result: dict,
    evidence_result: dict,
    story_mapping_result: dict,
    video_packaging_result: dict,
    script_generation_result: dict,
    metadata_creator_result: dict,
    voiceover_result=None,
) -> dict:
    summaries = {
        "news_collection": news_result.get("summary", {}),
        "evidence_prep": evidence_result.get("summary", {}),
        "story_mapping": story_mapping_result.get("summary", {}),
        "video_packaging": video_packaging_result.get("summary", {}),
        "script_generation": script_generation_result.get("summary", {}),
        "metadata_creator": metadata_creator_result.get("summary", {}),
    }
    if voiceover_result is not None:
        summaries["voiceover"] = voiceover_result.get("summary", {})
    return summaries


def build_draft_verification_package(
    script_generation_result: dict,
    metadata_creator_result: dict,
    video_packaging_result: dict,
    voiceover_result=None,
    channel=None,
    script_generation_input=None,
) -> dict:
    script_package = script_generation_result["data"]["script_package"]
    metadata_package = metadata_creator_result["data"]["metadata_package"]
    youtube_metadata = metadata_package.get("youtube_metadata", {})
    video_packet = video_packaging_result["data"]["video_packet"]
    sections = script_package.get("sections", [])
    visual_notes = [
        {
            "section_id": section.get("section_id", ""),
            "note": section.get("visual_note", ""),
        }
        for section in sections
        if section.get("visual_note")
    ]
    voiceover_package = None
    if voiceover_result is not None and voiceover_result.get("status") == "ok":
        voiceover_package = voiceover_result.get("data", {}).get("voiceover_package")

    script_tone = (script_generation_input or {}).get("config", {}).get("script_style", {}).get("tone", "")

    final_publish_package = {
        "package_type": "draft_v2_publish_package",
        "selected_format": video_packaging_result["data"]["selected_format"],
        "narration_text": script_package.get("full_script", ""),
        "youtube_metadata": youtube_metadata,
        "video": {
            "selected_format": video_packaging_result["data"]["selected_format"],
            "target_duration_seconds": video_packet.get("target_duration_seconds"),
            "story_count": video_packet.get("story_count"),
            "channel_name": (channel or {}).get("channel_name", "ManiVerse - AI"),
            "viewer_promise": youtube_metadata.get("description_intro", ""),
            "tone": script_tone,
        },
        "visual_notes": visual_notes,
        "source_context": {
            "selected_story_ids": video_packet.get("selected_story_ids", []),
            "stories": video_packet.get("stories", []),
        },
        "voiceover": voiceover_package,
        "evidence_boundaries": {
            "blocked_angles": youtube_metadata.get("blocked_title_angles", []),
        },
    }
    return {
        "status": "draft_unverified",
        "decision": {
            "ready_for_layer_9": False,
            "reason": "Pipeline v2 verification app has not been migrated yet.",
        },
        "final_publish_package": final_publish_package,
        "layer9_handoff": {"ready_for_layer_9": False},
    }


def build_chatgpt_handoff_input(
    run_id: str,
    manifest: dict,
    summaries: dict,
    story_mapping_result: dict,
    video_packaging_result: dict,
    script_generation_result: dict,
    draft_verification_package: dict,
    profile: str = "auto",
) -> dict:
    output_dir = HANDOFF_OUTPUT_DIR / "chatgpt" / profile
    return {
        "run": {"run_id": run_id, "generated_at": datetime.now(IST).isoformat()},
        "config": {
            "profile": profile,
            "prompt_version": "v1",
            "output_dir": str(output_dir),
        },
        "input": {
            "manifest": manifest,
            "artifacts": {
                "story_map": story_mapping_result.get("data", {}),
                "content_packet": video_packaging_result.get("data", {}).get("video_packet", {}),
                "script_review": script_generation_result,
                "publish_review": draft_verification_package,
            },
            "summaries": summaries,
        },
    }


def build_publish_handoff_input(run_id: str, manifest: dict, draft_verification_package: dict) -> dict:
    output_dir = HANDOFF_OUTPUT_DIR / "publish_preview"
    return {
        "run": {"run_id": run_id, "generated_at": datetime.now(IST).isoformat()},
        "config": {
            "output_dir": str(output_dir),
            "require_approved": False,
        },
        "input": {
            "manifest": manifest,
            "verification_package": draft_verification_package,
        },
    }


## App 1: News Collection

In [ ]:
from src.pipeline_v2.apps.news_collection import run as run_news_collection

news_input = build_news_collection_input(RUN_ID, RUN_PROFILE)
news_result = run_news_collection(news_input)
require_app_ready(news_result, "news_collection", allowed_statuses=("ok", "partial"))

write_json(NEWS_OUTPUT_DIR / "news_collection_result.json", news_result)
write_json(NEWS_OUTPUT_DIR / "raw_records.json", news_result["data"])
write_json(NEWS_OUTPUT_DIR / "summary.json", news_result["summary"])

print(f"News collection status: {news_result['status']}")
print(f"Raw records:            {news_result['data']['record_count']}")
print(f"Wrote:                  {NEWS_OUTPUT_DIR}")

In [ ]:
news_result["summary"]

## App 2: Evidence Prep

In [ ]:
from src.pipeline_v2.apps.evidence_prep import run as run_evidence_prep

evidence_input = build_evidence_prep_input(RUN_ID, news_input, news_result)
evidence_result = run_evidence_prep(evidence_input)
require_app_ready(evidence_result, "evidence_prep")

write_json(EVIDENCE_OUTPUT_DIR / "evidence_prep_result.json", evidence_result)
write_json(EVIDENCE_OUTPUT_DIR / "prepared_records.json", evidence_result["data"])
write_json(EVIDENCE_OUTPUT_DIR / "summary.json", evidence_result["summary"])

print(f"Evidence prep status: {evidence_result['status']}")
print(f"Prepared records:     {evidence_result['data']['record_count']}")
print(f"Wrote:                {EVIDENCE_OUTPUT_DIR}")

In [ ]:
evidence_result["summary"]

## App 3: Story Mapping

In [ ]:
# Optional story-mapping resume helper:
# Keep this commented for normal full-notebook runs.
#
# import json
# RESUME_RUN_ID = "v2_apps_YYYY_MM_DD_HHMMSS_daily_48h_all_sources"
# RUN_ID = RESUME_RUN_ID
# with open(PROJECT_ROOT / "output_v2" / "evidence_prep" / RUN_ID / "evidence_prep_result.json", encoding="utf-8") as f:
#     evidence_result = json.load(f)


In [ ]:
from src.pipeline_v2.apps.story_mapping import run as run_story_mapping

story_mapping_input = build_story_mapping_input(RUN_ID, evidence_input, evidence_result)
story_mapping_result = run_story_mapping(story_mapping_input)
require_app_ready(story_mapping_result, "story_mapping", handoff_key="ready_for_fact_packet")

write_json(STORY_OUTPUT_DIR / "story_mapping_result.json", story_mapping_result)
write_json(STORY_OUTPUT_DIR / "stories.json", story_mapping_result["data"])
write_json(STORY_OUTPUT_DIR / "summary.json", story_mapping_result["summary"])
write_jsonl(STORY_OUTPUT_DIR / "llm_calls.jsonl", story_mapping_result["llm_calls"])

print(f"Story mapping status: {story_mapping_result['status']}")
print(f"Stories:              {story_mapping_result['data']['story_count']}")
print(f"Wrote:                {STORY_OUTPUT_DIR}")

In [ ]:
story_mapping_result["summary"]

## App 4: Video Packaging

In [ ]:
# Optional video-packaging resume helper:
# Keep this commented for normal full-notebook runs.
#
# import json
# RESUME_RUN_ID = "v2_apps_YYYY_MM_DD_HHMMSS_daily_48h_all_sources"
# RUN_ID = RESUME_RUN_ID
# with open(PROJECT_ROOT / "output_v2" / "story_mapping" / RUN_ID / "story_mapping_result.json", encoding="utf-8") as f:
#     story_mapping_result = json.load(f)


In [ ]:
from src.pipeline_v2.apps.video_packaging import run as run_video_packaging

video_packaging_input = build_video_packaging_input(RUN_ID, story_mapping_result)
video_packaging_result = run_video_packaging(video_packaging_input)
require_app_ready(video_packaging_result, "video_packaging", handoff_key="ready_for_script_generation")

write_json(VIDEO_OUTPUT_DIR / "video_packaging_result.json", video_packaging_result)
write_json(VIDEO_OUTPUT_DIR / "video_packet.json", video_packaging_result["data"]["video_packet"])
write_json(VIDEO_OUTPUT_DIR / "summary.json", video_packaging_result["summary"])

print(f"Video packaging status: {video_packaging_result['status']}")
print(f"Selected format:        {video_packaging_result['data']['selected_format']}")
print(f"Selected stories:       {video_packaging_result['summary']['selected_story_count']}")
print(f"Ready for script gen:   {video_packaging_result['handoff']['ready_for_script_generation']}")
print(f"Wrote:                  {VIDEO_OUTPUT_DIR}")

In [ ]:
video_packaging_result["summary"]

## App 5: Script Generation


In [ ]:
# Optional resume helper:
# If you want to start from an existing video_packaging result, set RESUME_RUN_ID
# manually and load it here. Keep this commented for normal full-notebook runs.
#
# import json
# RESUME_RUN_ID = "v2_apps_YYYY_MM_DD_HHMMSS_daily_48h_all_sources"
# RUN_ID = RESUME_RUN_ID
# with open(PROJECT_ROOT / "output_v2" / "video_packaging" / RUN_ID / "video_packaging_result.json", encoding="utf-8") as f:
#     video_packaging_result = json.load(f)


In [ ]:
from src.pipeline_v2.apps.script_generation.formats.roundup import run as run_roundup_script_generation

script_generation_input = build_script_generation_input(RUN_ID, video_packaging_result)
script_generation_result = run_roundup_script_generation(script_generation_input)
require_non_empty_script(script_generation_result)
script_package = script_generation_result["data"]["script_package"]

write_json(SCRIPT_OUTPUT_DIR / "script_generation_result.json", script_generation_result)
write_json(SCRIPT_OUTPUT_DIR / "script_package.json", script_package)
write_json(SCRIPT_OUTPUT_DIR / "summary.json", script_generation_result["summary"])
write_jsonl(SCRIPT_OUTPUT_DIR / "llm_calls.jsonl", script_generation_result["llm_calls"])
(SCRIPT_OUTPUT_DIR / "full_script.txt").parent.mkdir(parents=True, exist_ok=True)
(SCRIPT_OUTPUT_DIR / "full_script.txt").write_text(script_package["full_script"], encoding="utf-8")

print(f"Script generation status: {script_generation_result['status']}")
print(f"Sections:                 {script_generation_result['summary']['section_count']}")
print(f"Words:                    {script_generation_result['summary']['total_word_count']}")
print(f"Estimated seconds:        {script_generation_result['summary']['estimated_duration_seconds']}")
print(f"Ready for verification:   {script_generation_result['handoff']['ready_for_verification']}")
print(f"Wrote:                    {SCRIPT_OUTPUT_DIR}")


In [ ]:
script_generation_result["summary"]


## App 6: Metadata Creator

In [ ]:
from src.pipeline_v2.apps.metadata_creator import run as run_metadata_creator

metadata_creator_input = build_metadata_creator_input(RUN_ID, script_generation_result, video_packaging_result)
metadata_creator_result = run_metadata_creator(metadata_creator_input)
require_app_ready(metadata_creator_result, "metadata_creator", handoff_key="ready_for_verification")
metadata_package = metadata_creator_result["data"]["metadata_package"]
youtube_metadata = metadata_package["youtube_metadata"]

write_json(METADATA_OUTPUT_DIR / "metadata_creator_result.json", metadata_creator_result)
write_json(METADATA_OUTPUT_DIR / "metadata_package.json", metadata_package)
write_json(METADATA_OUTPUT_DIR / "youtube_metadata.json", youtube_metadata)
write_json(METADATA_OUTPUT_DIR / "summary.json", metadata_creator_result["summary"])
write_jsonl(METADATA_OUTPUT_DIR / "llm_calls.jsonl", metadata_creator_result["llm_calls"])

print(f"Metadata status:          {metadata_creator_result['status']}")
print(f"Title options:            {metadata_creator_result['summary']['title_option_count']}")
print(f"Thumbnail options:        {metadata_creator_result['summary']['thumbnail_option_count']}")
print(f"Chapters:                 {metadata_creator_result['summary']['chapter_count']}")
print(f"Source URLs:              {metadata_creator_result['summary']['source_url_count']}")
print(f"Ready for verification:   {metadata_creator_result['handoff']['ready_for_verification']}")
print(f"Wrote:                    {METADATA_OUTPUT_DIR}")


In [ ]:
metadata_creator_result["summary"]

## App 7: Voiceover

Runs the app-like `src.pipeline_v2.voiceover` module through `.venv-voiceover`, because local TTS dependencies need the Python 3.12 voiceover environment. Audio behavior is controlled by `VOICEOVER_CONFIG` in the setup cell.


In [ ]:
# Optional voiceover-only resume helper:
# Keep this commented for normal full-notebook runs.
#
import json
RESUME_RUN_ID = "v2_apps_2026_06_07_234108_daily_48h_all_sources"
RUN_ID = RESUME_RUN_ID
with open(PROJECT_ROOT / "output_v2" / "script_generation" / RUN_ID / "script_generation_result.json", encoding="utf-8") as f:
    script_generation_result = json.load(f)


In [ ]:
import os
import subprocess

from src.pipeline_v2.voiceover.text_chunks import split_narration_into_chunks

script_text_for_preview = require_non_empty_script(script_generation_result)
voiceover_input = build_voiceover_input(RUN_ID, script_generation_result)
voiceover_output_dir = Path(voiceover_input["config"]["output_dir"])
voiceover_chunking = resolved_voiceover_chunking(voiceover_input["config"])
planned_chunks = split_narration_into_chunks(script_text_for_preview, **voiceover_chunking)
if not planned_chunks:
    raise RuntimeError("Voiceover preflight produced zero chunks from a non-empty script; check chunking config.")

print(f"Voiceover provider:       {voiceover_input['config']['provider']}")
print(f"Voiceover style:          {voiceover_input['config']['style']}")
print(f"Voiceover speed:          {voiceover_input['config'].get('speed')}")
print(f"Voiceover output:         {voiceover_output_dir}")
print(f"Resolved chunking:        {voiceover_chunking}")
print(f"Planned voiceover chunks: {len(planned_chunks)}")

VOICEOVER_INPUT_DIR.mkdir(parents=True, exist_ok=True)
voiceover_input_path = VOICEOVER_INPUT_DIR / f"{RUN_ID}_{voiceover_variant(voiceover_input['config'])}_voiceover_input.json"
write_json(voiceover_input_path, voiceover_input)

voiceover_python = PROJECT_ROOT / ".venv-voiceover" / "bin" / "python"
if not voiceover_python.exists():
    raise FileNotFoundError(f"Voiceover venv Python not found: {voiceover_python}")

env = {**os.environ, "PYTHONPATH": str(PROJECT_ROOT)}
subprocess.run(
    [
        str(voiceover_python),
        "-m",
        "src.pipeline_v2.voiceover.cli",
        "--input-json",
        str(voiceover_input_path),
    ],
    cwd=PROJECT_ROOT,
    env=env,
    check=True,
)

voiceover_summary = json.loads((voiceover_output_dir / "summary.json").read_text(encoding="utf-8"))
voiceover_package = json.loads((voiceover_output_dir / "voiceover_package.json").read_text(encoding="utf-8"))
voiceover_result = {
    "status": "ok",
    "data": {
        "run_id": RUN_ID,
        "generated_at": voiceover_package.get("generated_at"),
        "voiceover_package": voiceover_package,
    },
    "summary": voiceover_summary,
    "handoff": voiceover_package.get("handoff", {}),
}
write_json(voiceover_output_dir / "voiceover_input.json", voiceover_input)
write_json(voiceover_output_dir / "voiceover_result.json", voiceover_result)

print(f"Voiceover status: {voiceover_summary['status']}")
print(f"Voice:            {voiceover_summary['voice']}")
print(f"Duration seconds: {voiceover_summary['duration_seconds']}")
print(f"Chunks:           {voiceover_summary['chunk_count']}")
print(f"Audio path:       {voiceover_summary['audio_path']}")
print(f"Wrote:            {voiceover_output_dir}")


In [ ]:
voiceover_summary

## App 8: Handoffs

Creates final review handoffs from the v2 artifacts. Because v2 verification is not migrated yet, the publish handoff is intentionally marked as `draft_unverified` and should be treated as a human review preview, not an approved upload package.


In [ ]:
from src.pipeline_v2.handoffs.chatgpt_exporter import run as run_chatgpt_exporter
from src.pipeline_v2.handoffs.publish_handoff import run as run_publish_handoff

channel_context = news_input.get("input", {}).get("channel", {})
voiceover_result_for_handoff = globals().get("voiceover_result")

handoff_manifest = build_v2_manifest_context(
    RUN_ID,
    RUN_PROFILE,
    news_input,
    evidence_input,
    story_mapping_input,
    video_packaging_input,
    script_generation_input,
    metadata_creator_input,
)
summary_bundle = build_summary_bundle(
    news_result,
    evidence_result,
    story_mapping_result,
    video_packaging_result,
    script_generation_result,
    metadata_creator_result,
    voiceover_result_for_handoff,
)
draft_verification_package = build_draft_verification_package(
    script_generation_result,
    metadata_creator_result,
    video_packaging_result,
    voiceover_result=voiceover_result_for_handoff,
    channel=channel_context,
    script_generation_input=script_generation_input,
)

chatgpt_handoff_input = build_chatgpt_handoff_input(
    RUN_ID,
    handoff_manifest,
    summary_bundle,
    story_mapping_result,
    video_packaging_result,
    script_generation_result,
    draft_verification_package,
    profile="auto",
)
chatgpt_handoff_result = run_chatgpt_exporter(chatgpt_handoff_input)
if chatgpt_handoff_result["status"] != "ok":
    raise RuntimeError(chatgpt_handoff_result["summary"])

publish_handoff_input = build_publish_handoff_input(RUN_ID, handoff_manifest, draft_verification_package)
publish_handoff_result = run_publish_handoff(publish_handoff_input)
if publish_handoff_result["status"] != "ok":
    raise RuntimeError(publish_handoff_result["summary"])

chatgpt_output_dir = Path(chatgpt_handoff_input["config"]["output_dir"])
publish_output_dir = Path(publish_handoff_input["config"]["output_dir"])
chatgpt_paths = write_handoff_files(chatgpt_handoff_result["data"]["handoff_package"]["files"], chatgpt_output_dir)
publish_paths = write_handoff_files(publish_handoff_result["data"]["publish_handoff_package"]["files"], publish_output_dir)

write_json(HANDOFF_OUTPUT_DIR / "manifest_context.json", handoff_manifest)
write_json(HANDOFF_OUTPUT_DIR / "summary_bundle.json", summary_bundle)
write_json(HANDOFF_OUTPUT_DIR / "draft_verification_package.json", draft_verification_package)
write_json(HANDOFF_OUTPUT_DIR / "chatgpt_handoff_input.json", chatgpt_handoff_input)
write_json(HANDOFF_OUTPUT_DIR / "chatgpt_handoff_result.json", chatgpt_handoff_result)
write_json(HANDOFF_OUTPUT_DIR / "publish_handoff_input.json", publish_handoff_input)
write_json(HANDOFF_OUTPUT_DIR / "publish_handoff_result.json", publish_handoff_result)

print(f"ChatGPT handoff status: {chatgpt_handoff_result['status']}")
print(f"ChatGPT profile:        {chatgpt_handoff_result['summary']['profile']}")
print(f"ChatGPT files:          {len(chatgpt_paths)} -> {chatgpt_output_dir}")
print(f"Publish handoff status: {publish_handoff_result['status']}")
print(f"Publish files:          {len(publish_paths)} -> {publish_output_dir}")
print(f"Wrote handoff context:  {HANDOFF_OUTPUT_DIR}")


In [ ]:
{
    "chatgpt_handoff": chatgpt_handoff_result["summary"],
    "publish_handoff": publish_handoff_result["summary"],
    "draft_verification_status": draft_verification_package["status"],
}
